# 🦜 AIFlow — Train Custom Voice (Colab)

Train hoặc encode 1 giọng custom để dùng trong AIFlow. Output là 1 file `.zip` import qua **Voice Gallery → Import**.

## 🎯 2 path

| Path | Khi nào | Time | Output |
|------|---------|------|--------|
| **A — LoRA fine-tune** | ≥30 phút audio sạch | 5-60 phút (theo GPU) | ~50 MB zip |
| **B — Persistent embedding** | 3-15s ref | <1 phút | ~15 KB zip |

## 🖥️ Colab runtime

| Runtime | Path A train (3000 steps) |
|---------|--------------------------|
| CPU only | ❌ Path A — chỉ Path B |
| T4 (free) | ~25-30 phút |
| L4 (Pro) | ~12-15 phút |
| A100 40GB (Pro) | ~5-8 phút |
| A100 80GB / H100 (Pro+) | ~3-5 phút |
| TPU | ❌ không hỗ trợ |

**Runtime → Change runtime type** → chọn GPU phù hợp gói Colab.

## 🛠️ Cấu trúc notebook

Mỗi cell `%run` 1 file `.py` riêng trong `cells/`. Edit logic trong file `.py`,
notebook chỉ là vỏ điều phối.

```
cells/
├── _shared.py           ← state + helpers chung
├── 01a_runtime_detect.py
├── 01b_drive_mount.py
├── ...
└── 06c_download_export.py
```

Xem [`cells/README.md`](cells/README.md) để biết chi tiết từng cell.

## 📚 Refs

- Spec đầy đủ: `app/docs/11-custom-voice-training.md`
- VieNeu-TTS: <https://huggingface.co/pnnbao-ump/VieNeu-TTS-v2>

## 🪜 Setup notebook (chạy lần đầu mỗi session)

Notebook + folder `cells/` phải nằm cùng repo trên GitHub. Cell dưới clone repo của bạn xuống Colab và `cd` vào `app/colab/` để mọi `%run cells/...` tham chiếu đúng path.

**Sửa `REPO_URL` thành GitHub URL của bạn** trước khi chạy:

In [ ]:
# 👉 SỬA URL này thành repo AIFlow của bạn
REPO_URL = 'https://github.com/yourname/aiflow.git'

import os, subprocess, sys
from pathlib import Path

AIFLOW_DIR = Path('/content/aiflow')
if not AIFLOW_DIR.exists():
    print(f'📥 Cloning {REPO_URL}...')
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, str(AIFLOW_DIR)], check=True)
else:
    print('✅ Repo đã có sẵn')

COLAB_DIR = AIFLOW_DIR / 'app' / 'colab'
os.chdir(COLAB_DIR)
sys.path.insert(0, str(COLAB_DIR))

print(f'✅ Working dir: {COLAB_DIR}')
print(f'   Cells dir:   {COLAB_DIR / "cells"}')

# Verify cells folder tồn tại
if not (COLAB_DIR / 'cells' / '_shared.py').exists():
    print('❌ Không tìm thấy cells/_shared.py — repo chưa đúng cấu trúc.')

---
## 📍 Phần 1 — Setup môi trường
Chạy 1a → 1f tuần tự.

### 1a — Detect runtime

In [ ]:
%run cells/01a_runtime_detect.py

### 1b — Mount Google Drive

In [ ]:
%run cells/01b_drive_mount.py

### 1c — Clone VieNeu-TTS repo

In [ ]:
%run cells/01c_clone_repo.py

### 1d — Install dependencies (~3-5 phút lần đầu)

In [ ]:
%run cells/01d_install_deps.py

### 1e — Setup paths + HF cache

In [ ]:
%run cells/01e_setup_paths.py

### 1f — Init work dir

In [ ]:
%run cells/01f_init_state.py

---
## 📍 Phần 2 — Upload data
Chọn 1 trong 2 path:
- **Path A**: chạy 2b (zip dataset cho LoRA fine-tune)
- **Path B**: chạy 2c (1 file ref audio cho persistent embedding)

Sau đó chạy 2d để verify.

### 2b — Path A: Upload zip dataset

In [ ]:
%run cells/02b_upload_path_a.py

### 2c — Path B: Upload 1 file ref audio

In [ ]:
%run cells/02c_upload_path_b.py

### 2d — Verify state

In [ ]:
%run cells/02d_verify_state.py

---
## 📍 Phần 3 — Config voice
Chạy 3a → 3e tuần tự. 3b chỉ cần cho Path B, 3c+3d chỉ cần cho Path A.

### 3a — Voice metadata widgets

In [ ]:
%run cells/03a_voice_metadata.py

### 3b — Path B: ref text

In [ ]:
%run cells/03b_ref_text_input.py

### 3c — Path A: quality preset (GPU-aware)

In [ ]:
%run cells/03c_quality_preset.py

### 3d — Path A: advanced override (chỉ khi chọn 'advanced' ở 3c)

In [ ]:
%run cells/03d_advanced_override.py

### 3e — Save config

In [ ]:
%run cells/03e_save_config.py

---
## 📍 Phần 4 — Train / Encode

**Path B** (embedding): chạy 1 cell `04_path_b_encode.py`.

**Path A** (LoRA): chạy 4a → 4h tuần tự.

### Path B — encode embedding (~10s)

In [ ]:
%run cells/04_path_b_encode.py

### 4a — Path A: filter dataset

In [ ]:
%run cells/04a_filter_dataset.py

### 4b — Path A: encode dataset (~5-10 phút)

In [ ]:
%run cells/04b_encode_dataset.py

### 4c — Path A: load base model

In [ ]:
%run cells/04c_load_base_model.py

### 4d — Path A: setup LoRA

In [ ]:
%run cells/04d_setup_lora.py

### 4e — Path A: train (resume-safe via Drive checkpoint)

In [ ]:
%run cells/04e_train_loop.py

### 4f — Path A: save adapter + log

In [ ]:
%run cells/04f_save_adapter.py

### 4g — Path A: build voices.json

In [ ]:
%run cells/04g_build_voices_json.py

### 4h — Path A: cleanup VRAM trước khi test

In [ ]:
%run cells/04h_cleanup_vram.py

---
## 📍 Phần 5 — Test synthesis
Nghe thử voice trước khi export.

### 5a — Load engine

In [ ]:
%run cells/05a_load_engine.py

### 5b — Test synthesis (chạy nhiều lần với text khác)

In [ ]:
%run cells/05b_test_synthesis.py

---
## 📍 Phần 6 — Export package

### 6a — Build metadata.json + README

In [ ]:
%run cells/06a_build_metadata.py

### 6b — Pack zip + Drive backup

In [ ]:
%run cells/06b_pack_zip.py

### 6c — Download về máy

In [ ]:
%run cells/06c_download_export.py

---
## 🆘 Troubleshooting

**OOM lúc train (4e)**
- Giảm preset xuống 1 bậc ở Cell 3c
- Hoặc dùng Advanced ở 3c → 3d → giảm `batch_size` 2 → 1, tăng `grad_accum` để giữ effective batch
- Restart Runtime → Run All

**Colab disconnect giữa chừng**
- Reconnect → re-run từ Cell setup notebook trên cùng → 1a → ... → 4e
- Trainer auto resume từ checkpoint Drive (save_steps=500)

**Voice output không giống ref**
- Path A: tăng preset (Conservative → Balanced → Fast). Hoặc tăng `max_steps` ở Advanced
- Path B: ref audio quá ngắn (<3s) hoặc nhiễu. Thu lại

**voices.json import fail vào AIFlow**
- Check zip có `metadata.json` ở root + đúng `spec="aiflow.custom_voice"`
- Voice ID trong metadata phải match key trong voices.json `presets`

**"Module not found" lúc `%run cells/...`**
- Re-run cell setup notebook trên cùng (cd vào `app/colab/`)
- Đảm bảo đã clone repo đúng URL